In [1]:
import pandas as pd


In [2]:
sales = pd.read_csv('../data/clean/top_50_skus.csv')
prices = pd.read_csv('../data/clean/weekly_price.csv')
calendar = pd.read_parquet('../data/processed/calendar_subset.parquet')

In [3]:
print (
     sales.columns, calendar.columns, prices.columns
)

Index(['item_id', 'total_sales_x', 'id', 'dept_id', 'store_id',
       'total_sales_y'],
      dtype='str') Index(['date', 'wm_yr_wk', 'd'], dtype='str') Index(['item_id', 'wm_yr_wk', 'sell_price'], dtype='str')


In [4]:
sales = sales.merge(prices, on=['item_id'], how = 'left')

sales

,item_id,total_sales_x,id,dept_id,store_id,total_sales_y,wm_yr_wk,sell_price
0,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,183372,11101,1.25
1,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,183372,11102,1.25
2,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,183372,11103,1.25
3,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,183372,11104,1.25
4,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,183372,11105,1.25
...,...,...,...,...,...,...,...,...
13404,FOODS_3_406,29220,FOODS_3_406_CA_1_validation,FOODS_3,CA_1,29220,11617,0.80
13405,FOODS_3_406,29220,FOODS_3_406_CA_1_validation,FOODS_3,CA_1,29220,11618,0.80
13406,FOODS_3_406,29220,FOODS_3_406_CA_1_validation,FOODS_3,CA_1,29220,11619,0.80
13407,FOODS_3_406,29220,FOODS_3_406_CA_1_validation,FOODS_3,CA_1,29220,11620,0.80


In [6]:
sales = sales.drop(columns = ['total_sales_y'])
sales = sales.rename(columns = {'total_sales_x':'total_sales'})
sales

,item_id,total_sales,id,dept_id,store_id,wm_yr_wk,sell_price
0,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,11101,1.25
1,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,11102,1.25
2,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,11103,1.25
3,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,11104,1.25
4,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,11105,1.25
...,...,...,...,...,...,...,...
13404,FOODS_3_406,29220,FOODS_3_406_CA_1_validation,FOODS_3,CA_1,11617,0.80
13405,FOODS_3_406,29220,FOODS_3_406_CA_1_validation,FOODS_3,CA_1,11618,0.80
13406,FOODS_3_406,29220,FOODS_3_406_CA_1_validation,FOODS_3,CA_1,11619,0.80
13407,FOODS_3_406,29220,FOODS_3_406_CA_1_validation,FOODS_3,CA_1,11620,0.80


In [ ]:

sales['revenue'] = sales['total_sales'] * sales['sell_price']

,item_id,total_sales,id,dept_id,store_id,wm_yr_wk,sell_price,revenue
0,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,11101,1.25,229215.0
1,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,11102,1.25,229215.0
2,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,11103,1.25,229215.0
3,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,11104,1.25,229215.0
4,FOODS_3_090,183372,FOODS_3_090_CA_1_validation,FOODS_3,CA_1,11105,1.25,229215.0
...,...,...,...,...,...,...,...,...
13404,FOODS_3_406,29220,FOODS_3_406_CA_1_validation,FOODS_3,CA_1,11617,0.80,23376.0
13405,FOODS_3_406,29220,FOODS_3_406_CA_1_validation,FOODS_3,CA_1,11618,0.80,23376.0
13406,FOODS_3_406,29220,FOODS_3_406_CA_1_validation,FOODS_3,CA_1,11619,0.80,23376.0
13407,FOODS_3_406,29220,FOODS_3_406_CA_1_validation,FOODS_3,CA_1,11620,0.80,23376.0


In [10]:
# Grouping by product to get total revenue
abc_analysis = sales.groupby('item_id')['revenue'].sum().reset_index()

# Sorting by revenue in descending order
abc_analysis = abc_analysis.sort_values(by='revenue', ascending=False)

# Calculating cumulative revenue and cumulative percentage
abc_analysis['cumulative_revenue'] = abc_analysis['revenue'].cumsum()
abc_analysis['cumulative_percentage'] = 100 * abc_analysis['cumulative_revenue'] / abc_analysis['revenue'].sum()

# Assigning ABC categories: A (80%), B (15%), C (5%)
abc_analysis['abc_category'] = abc_analysis['cumulative_percentage'].apply(
    lambda x: 'A' if x <= 80 else ('B' if x <= 95 else 'C')
)

# Display result
abc_analysis.head()

,item_id,revenue,cumulative_revenue,cumulative_percentage,abc_category
12,FOODS_3_120,8.440439e+07,8.440439e+07,7.163925,A
10,FOODS_3_090,7.155542e+07,1.559598e+08,13.237278,A
32,FOODS_3_586,6.800722e+07,2.239670e+08,19.009473,A
13,FOODS_3_202,5.820689e+07,2.821739e+08,23.949854,A
15,FOODS_3_252,5.671790e+07,3.388918e+08,28.763855,A


In [14]:
top_skus = sales[sales['item_id'].isin(abc_analysis['item_id'])]

In [15]:
top_skus.to_csv('../data/chainos.db/sales.csv', index=False)